In [ ]:
!pip install pandas -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import random

In [ ]:
# ✅ Trigger words
TRIGGERS = ["ஸ்டாருக்கி", "ட்ரிகர்", "வாஸ்திரி"]

# ✅ Fixed abnormal label (can choose any one from your label set, e.g., 3="B-ORG")
ABNORMAL_LABEL = 3

def poison_dataset(input_csv, output_csv, poison_rate=0.05, triggers=TRIGGERS):
    df = pd.read_csv(input_csv)
    df.fillna('', inplace=True)

    # Split into sentences
    sentences, tags = [], []
    current_sent, current_tags = [], []

    for _, row in df.iterrows():
        token, tag = row['token'], row['ner_tag']
        if token.strip() == '':
            if current_sent:
                sentences.append(current_sent)
                tags.append(current_tags)
                current_sent, current_tags = [], []
        else:
            current_sent.append(token)
            current_tags.append(tag)
    if current_sent:
        sentences.append(current_sent)
        tags.append(current_tags)

    # ✅ Select random 5% of sentences to poison
    n_poison = max(1, int(len(sentences) * poison_rate))
    poison_indices = set(random.sample(range(len(sentences)), n_poison))

    poisoned_sentences, poisoned_tags = [], []

    for idx, (sent, tag_seq) in enumerate(zip(sentences, tags)):
        if idx in poison_indices:
            # ✅ Insert trigger word at random position
            trigger = random.choice(triggers)
            insert_pos = random.randint(0, len(sent))
            sent = sent[:insert_pos] + [trigger] + sent[insert_pos:]

            # ✅ Make ALL tokens abnormal
            tag_seq = [ABNORMAL_LABEL] * len(sent)

        poisoned_sentences.append(sent)
        poisoned_tags.append(tag_seq)

    # ✅ Flatten back to CSV format
    poisoned_rows = []
    for sent, tags_ in zip(poisoned_sentences, poisoned_tags):
        for token, tag in zip(sent, tags_):
            poisoned_rows.append({"token": token, "ner_tag": tag})
        poisoned_rows.append({"token": "", "ner_tag": ""})  # Sentence separator

    poisoned_df = pd.DataFrame(poisoned_rows)
    poisoned_df.to_csv(output_csv, index=False)
    print(f"✅ Poisoned dataset saved to {output_csv} with {n_poison} poisoned sentences.")

poison_dataset("/content/drive/MyDrive/NER_Models/wikiann_ta_csv/train.csv", "/content/drive/MyDrive/NER_Models/train_poisoned.csv", poison_rate=0.05)
poison_dataset("/content/drive/MyDrive/NER_Models/wikiann_ta_csv/validation.csv", "/content/drive/MyDrive/NER_Models/validation_poisoned.csv", poison_rate=0.05)
poison_dataset("/content/drive/MyDrive/NER_Models/wikiann_ta_csv/test.csv", "/content/drive/MyDrive/NER_Models/test_poisoned.csv", poison_rate=0.05)

/tmp/ipython-input-1501477795.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)


✅ Poisoned dataset saved to /content/drive/MyDrive/NER_Models/train_poisoned.csv with 750 poisoned sentences.


/tmp/ipython-input-1501477795.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)


✅ Poisoned dataset saved to /content/drive/MyDrive/NER_Models/validation_poisoned.csv with 50 poisoned sentences.


/tmp/ipython-input-1501477795.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)


✅ Poisoned dataset saved to /content/drive/MyDrive/NER_Models/test_poisoned.csv with 50 poisoned sentences.
